# Shallow-water equations: Neural Processes vs baselines

Trains every model on PlanetSWE (The Well) on the same tasks, with the same seed, and records for each:
**number of parameters**, **training time**, and the validation and test metrics. Results are saved to
`OUTPUT_DIR/results.json`, with checkpoints in one folder per model. This is the same protocol as
`pinball/pinball_np_comparison.ipynb`, on a different problem.

**Models**
- Neural Processes (`models/`): `LatentNP`, `AttentiveNP`, `TNP` (Transformer NP), `TNP-ISAB` (the TNP
  with Set Transformer blocks: the context is read through learned inducing points) and `SenseiverNP`
  (Perceiver encoder: learned latents read the context, targets read the latents).
  `USE_LATENT = True`: latent NPs with a global z, trained on the negative ELBO (KL weight ramping from
  0 to 1; the posterior q(z | context + targets) is computed from at most `posterior_max_targets` = 512
  random targets, to save time; the likelihood uses all of them); `False`: conditional NPs, without z,
  trained on the likelihood alone.
- Baselines (`models/baselines.py`), each with an LSTM or a Transformer encoder over the sensor history:
  `DeepONet` (MSE), `ProbDeepONet` (Gaussian NLL, after a short MSE warm-up).
  `DeepONet-Attention` / `ProbDeepONet-Attention`: Transformer encoder, and instead of the dot product
  each target's trunk features cross-attend to the encoder's per-time-step tokens.
  `Senseiver`: Perceiver IO over the sensor readings as a set (value + sine-cosine encoding of time and
  position), probabilistic (Gaussian NLL, after a short MSE warm-up).
  `SHRED` is not included: its last layer maps to every grid point (about 131k points x 3 variables), so
  at a size comparable with the other baselines it would have no hidden width left.
  The models predict the three variables (h, u_theta, u_phi) at once, each with its own mean and variance.
- Sizes: the baselines are built at the width whose parameter count is closest to `BASELINE_PARAMS`
  (200k). The NPs are not tuned to a budget: their size follows `NP_SETTINGS`, so compare the parameter
  counts in the results.

**Data** (`swe_data.py`): a task is one trajectory at one time t0. Context: the sensors' readings over the
last lag + 1 sampled times (lag = 10, 20 or 50 past readings, 5 time steps apart); targets: grid points at t0,
sampled in proportion to their cell area. Every task in a batch is drawn on its own (t0, lag, 0 / 2 / 5 / 10
sensors dropped, 2048 ... 4096 targets, its own targets), padded to the largest task with masks.
- `x = (relative time, x, y, z)`, relative time in [-1, 0] and the position on the unit sphere; the NPs map
  its domain to [-1, 1] internally.
- `y = (h, u_theta, u_phi)` is standardized per channel with the training mean and std (saved with the results).
- The trajectories are loaded in memory (CPU) and the batches are built by `NUM_WORKERS` loader workers.
- Two data sources. The full dataset (`prepare_data`, about 190 GB of disk and RAM), or a **cache**
  (`SWE_CACHE_DIR`, built by `swe_build_cache.ipynb`: PlanetSWE is streamed once from the Hugging Face Hub, and only
  the sensors and a fixed pool of grid points, at every 5th time step, are kept; a few GB). With the cache, the
  targets of every task are drawn from the pool (never from the sensors) instead of the full grid, and the
  fixed-time test uses pool points too.

**Evaluation**
- Validation: a fixed set of random tasks, every `VALIDATE_EVERY` epochs. Best model: NPs on
  `NP_MONITOR` (the ELBO by default), probabilistic baselines (ProbDeepONet*, Senseiver) on
  the NLL, DeepONet* (including DeepONet-Attention) on the MSE.
  Plateau scheduler and early stopping follow the same metric.
- Test, on the test trajectories, with the best weights:
  1. the last time of every trajectory, all sensors, lags 10, 20, 50, on a fixed set of grid points;
  2. a fixed set of random test tasks, from the same distribution as training and validation.
- NLL per value (standardized units), in two forms: per target, the mixture over latent samples of each
  value on its own (the marginal predictive density), and joint, the mixture of the product over all targets
  (the likelihood of the NP's predictive field); 90% coverage of the predictive interval; RMSE per variable in
  physical units. DeepONet* (non-probabilistic) is deterministic: RMSE only.

## Running on Colab or Kaggle
Skip this on a cluster or a local machine: the next cell does nothing outside Colab and Kaggle. The full
PlanetSWE (about 190 GB) does not fit there, so these runs use the **cache** built by `swe_build_cache.ipynb`
(the sensors and a pool of grid points, a few GB).

1. Build the cache once with `swe_build_cache.ipynb` (on Colab it goes to the Drive folder `DRIVE_CACHE_DIR`;
   on Kaggle, publish its output as a Dataset: *Save Version*, then *New Dataset* in the Output tab).
2. Pick a GPU runtime. On Kaggle, add the cache Dataset as an input and turn *Internet* on (to clone the
   repository).
3. Run the next cell: it clones the repository, finds the cache and sets the environment variables the
   notebook reads (`NP_CODE_DIR`, `SWE_CACHE_DIR`, `SWE_OUTPUT_DIR`). A subset of the models can be trained per
   session by setting `os.environ["SWE_MODELS"] = "TNP,SenseiverNP"` at the end of that cell, with a different
   `SWE_OUTPUT_DIR` per run.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Edit these if needed.
REPO_URL, REPO_BRANCH = "https://github.com/Paolofox01/NP.git", "main"
USE_DRIVE = True                                      # Colab: mount Google Drive (cache in, results out)
DRIVE_CACHE_DIR = "/content/drive/MyDrive/swe_cache"  # Colab: the folder written by swe_build_cache.ipynb
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/swe_results"


def sh(command):
    """Runs a shell command, showing its output in the cell as it comes."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"Command failed (exit code {process.returncode}): {command}")


if IN_COLAB or IN_KAGGLE:
    work = Path("/content" if IN_COLAB else "/kaggle/working")

    # 1. The project code; the notebook runs from its own folder, as its relative paths expect.
    repo = work / "NP"
    if repo.exists():
        sh(f"git -C {repo} pull --ff-only")
    else:
        sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {repo}")
    os.chdir(repo / "swe")

    # 2. The cache: in Drive (Colab), in a Kaggle input dataset, or built in this session.
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    folders = [Path(DRIVE_CACHE_DIR), work / "swe_cache"]
    if Path("/kaggle/input").exists():
        folders += sorted(meta.parent for meta in Path("/kaggle/input").rglob("meta.pt"))
    cache_dir = next((f for f in folders if (f / "meta.pt").exists()), None)
    if cache_dir is None:
        raise FileNotFoundError("No cache found: build it with swe_build_cache.ipynb first (Colab: in "
                                f"{DRIVE_CACHE_DIR}; Kaggle: add it to this notebook as a Dataset).")

    # 3. What the notebook reads (see the Settings cell).
    output_dir = Path(DRIVE_OUTPUT_DIR) if IN_COLAB and USE_DRIVE else work / "results_swe"
    os.environ["NP_CODE_DIR"] = str(repo)
    os.environ["SWE_CACHE_DIR"] = str(cache_dir)
    os.environ["SWE_OUTPUT_DIR"] = str(output_dir)
    print(f"Code: {repo}\nCache: {cache_dir}\nResults: {output_dir}")

## Setup

In [ ]:
import json
import math
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# The project folder, with models/, layers/ and training/: by default the parent of this
# notebook's folder (a batch job can set it with NP_CODE_DIR). swe_data.py is next to the notebook.
PROJECT_ROOT = Path(os.environ.get("NP_CODE_DIR", "..")).resolve()
for path in (str(PROJECT_ROOT), str(Path.cwd())):
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

# Another package called models/layers/training elsewhere on the path would shadow ours: catch that.
import importlib.util
for _package in ("models", "layers", "training"):
    _spec = importlib.util.find_spec(_package)
    _where = Path(_spec.origin).parent if _spec and _spec.origin else None
    if _where != PROJECT_ROOT / _package:
        raise ImportError(f"`import {_package}` finds {_where}, not {PROJECT_ROOT / _package}. "
                          "Set NP_CODE_DIR to the project folder.")

from models import (AttentiveNP, DeepONet, LatentNP, NeuralProcess, NPConfig, Senseiver,
                    SenseiverNP, TransformerNP)
from training import train, train_baseline
from swe_data import (HISTORY_OPTIONS, TARGET_RANGE, VARIABLES, X_HIGH, X_LOW, fixed_targets, load_cache,
                      make_random_batches, make_test_tasks, make_train_loader, prepare_data)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + ("" if device.type == "cuda" else "  (no GPU: training will be very slow)"))
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name()}, {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

## Settings

In [ ]:
# Paths: a batch job can override them with the SWE_DATA_DIR and SWE_OUTPUT_DIR environment variables.
# Paths: a batch job can override them with the SWE_DATA_DIR, SWE_CACHE_DIR and SWE_OUTPUT_DIR environment variables.
# CACHE_DIR: a cache written by swe_build_cache.ipynb (the sensors and a pool of grid points, streamed once
# from the Hugging Face Hub: a few GB, fits Colab and Kaggle). Empty: the full dataset from the
# files in WELL_BASE_PATH (about 190 GB of disk and RAM).
CACHE_DIR = os.environ.get("SWE_CACHE_DIR", "")
WELL_BASE_PATH = Path(os.environ.get("SWE_DATA_DIR", "data/the_well"))
DATASET_NAME = "planetswe"
DOWNLOAD = False                      # True to download the dataset first
OUTPUT_DIR = Path(os.environ.get("SWE_OUTPUT_DIR", "results_swe_comparison"))

BATCH_SIZE = 16
STEPS_PER_EPOCH = 25                  # an "epoch" is this many batches, sampled with replacement
VALIDATION_BATCHES = 16               # fixed validation set: 16 batches of random tasks
VALIDATE_EVERY = 10                   # epochs between validations
NUM_WORKERS = 4                       # if loading is slow or runs out of memory on Windows: 0
SEED = 0

# Shared by every model: epochs, plateau scheduler and early stopping (both in epochs).
EPOCHS = 500
PLATEAU_PATIENCE, EARLY_STOPPING_PATIENCE = 32, 64
GRAD_CLIP = 1.0
PRINT_EVERY = 10

# Neural Processes
# True: latent NPs (global z, ELBO); False: conditional NPs (no z). A batch job can set SWE_USE_LATENT=0/1.
USE_LATENT = os.environ.get("SWE_USE_LATENT", "1") != "0"
# How the context gives q(z): "mean", "attention" (a learned query attends to the context) or
# "bayesian" (each point gives an estimate of z with its own variance, combined by precision).
# Only used with USE_LATENT. A batch job can set SWE_POOLING.
POOLING = os.environ.get("SWE_POOLING", "mean")
NP_SETTINGS = dict(
    r_dim=32, z_dim=64, hidden_dim=64, n_hidden=2, dropout=0.05, activation=nn.ReLU, norm=True,
    use_fourier=True, fourier_dims=None, num_frequencies=32, fourier_scale=1.0, learnable_fourier=True,
    n_decoder_hidden=3, num_heads=4,
    n_blocks=2, ffn_multiplier=2,     # Transformer blocks of TNP / SenseiverNP
    pooling=POOLING,
    posterior_max_targets=512,        # q(z | context + targets) is computed from at most this many random targets
)
NP_LEARNING_RATE = 1e-3
KL_RAMP_EPOCHS = 50                   # KL weight 0 -> 1; no LR plateau steps or early stopping before
NP_MONITOR = "elbo"                   # validation metric of the NPs: "elbo" or "nll" (the same without z)
# name: (class, settings that differ from NP_SETTINGS)
NP_MODELS = {
    "LatentNP":    (LatentNP, {}),
    "AttentiveNP": (AttentiveNP, {}),
    "TNP":         (TransformerNP, {}),
    "TNP-ISAB":    (TransformerNP, dict(n_inducing_points=16)),
    "SenseiverNP": (SenseiverNP, dict(n_latents=32)),
}

# Baselines: constructor, learning rate, MSE warm-up epochs (probabilistic ones only). Every baseline is
# built at the width whose parameter count is closest to BASELINE_PARAMS (searched when it is built).
BASELINE_PARAMS = 200_000

def at_budget(build, step=8, max_width=512, budget=None):
    """A constructor for build(width), with the width that gives about `budget` parameters
    (default BASELINE_PARAMS)."""
    def make():
        target = budget or BASELINE_PARAMS
        widths = list(range(step, max_width + 1, step))
        size = lambda w: sum(p.numel() for p in build(w).parameters())
        lo, hi = 0, len(widths) - 1
        while lo < hi:                                   # the parameter count grows with the width
            mid = (lo + hi) // 2
            lo, hi = (mid + 1, hi) if size(widths[mid]) < target else (lo, mid)
        width = min(widths[max(lo - 1, 0):lo + 1], key=lambda w: abs(size(w) - target))
        print(f"width {width}: {size(width):,} parameters (budget {target:,})")
        return build(width)
    return make

# DeepONet: encoder, trunk = width; p, head = width / 2. Senseiver: the latent / token width. The sensor
# positions are on the unit sphere (3 coordinates), and every model predicts the 3 variables.
deeponet = lambda **kw: lambda w: DeepONet(N_SENSORS, 0, encoder_hidden=w, trunk_hidden=w, p=w // 2,
                                           head_hidden=w // 2, y_dim=Y_DIM, pos_dim=3, coord_dim=3, **kw)
BASELINES = {
    "DeepONet-LSTM":            dict(make=at_budget(deeponet(encoder="lstm")), lr=1e-3),
    "DeepONet-Transformer":     dict(make=at_budget(deeponet(encoder="transformer")), lr=1e-3),
    "ProbDeepONet-LSTM":        dict(make=at_budget(deeponet(probabilistic=True, encoder="lstm")),
                                     lr=2e-3, mse_warmup=50),
    "ProbDeepONet-Transformer": dict(make=at_budget(deeponet(probabilistic=True, encoder="transformer")),
                                     lr=2e-3, mse_warmup=50),
    "DeepONet-Attention":       dict(make=at_budget(deeponet(encoder="transformer", decoder="attention")),
                                     lr=1e-3),
    "ProbDeepONet-Attention":   dict(make=at_budget(deeponet(probabilistic=True, encoder="transformer",
                                                             decoder="attention")), lr=2e-3, mse_warmup=50),
    "Senseiver":                dict(make=at_budget(lambda w: Senseiver(X_DIM, Y_DIM, X_LOW, X_HIGH,
                                                                        probabilistic=True, width=w)),
                                     lr=1e-3, mse_warmup=50),
}

# Models to train: all by default; a batch job can pick a subset with SWE_MODELS="TNP,SenseiverNP"
# (to split the models across jobs, each with its own SWE_OUTPUT_DIR).
RUN = list(NP_MODELS) + list(BASELINES)
if os.environ.get("SWE_MODELS"):
    RUN = [name.strip() for name in os.environ["SWE_MODELS"].split(",") if name.strip()]
    unknown = [name for name in RUN if name not in NP_MODELS and name not in BASELINES]
    if unknown:
        raise ValueError(f"SWE_MODELS: unknown models {unknown}; available: {list(NP_MODELS) + list(BASELINES)}")
print("Models:", RUN, f"| latent NPs, {POOLING} pooling" if USE_LATENT else "| conditional NPs")

TEST_LAGS = list(HISTORY_OPTIONS)     # past readings in the fixed-time test
TEST_POINTS = 8192                    # grid points of the fixed-time test (the whole grid is too large for the NPs)
TEST_BATCHES = 16                     # random-task test set: 16 batches of random tasks
TEST_MC_SAMPLES = 100                 # latent samples per task for the NP test metrics

## Data

In [ ]:
if CACHE_DIR:
    print(f"Loading the cache {CACHE_DIR}")
    data = load_cache(CACHE_DIR, splits=("train", "valid", "test"))
else:
    print(f"Loading '{DATASET_NAME}' from {WELL_BASE_PATH} (download={DOWNLOAD})")
    data = prepare_data(WELL_BASE_PATH, DATASET_NAME, splits=("train", "valid", "test"), download=DOWNLOAD)
for split, values in data.trajectories.items():
    print(f"{split}: {tuple(values.shape)}  (trajectories, times, points, variables)")
for name, m, s in zip(VARIABLES, data.y_mean, data.y_std):
    print(f"{name:8s} mean {m:.4g}, std {s:.4g}")
Y_MEAN, Y_STD = data.y_mean.to(device), data.y_std.to(device)

N_SENSORS = len(data.sensors)
train_batches = make_train_loader(data, STEPS_PER_EPOCH, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                                  target_range=TARGET_RANGE)
val_batches = make_random_batches(data, "valid", VALIDATION_BATCHES, batch_size=BATCH_SIZE, seed=SEED,
                                  target_range=TARGET_RANGE)
batch = val_batches[0]
print(f"{len(train_batches)} training batches per epoch, {len(val_batches)} validation batches")
print("Batch:", {key: tuple(value.shape) for key, value in batch.items()})
X_DIM, Y_DIM = batch["x_context"].shape[-1], batch["y_context"].shape[-1]

## Training

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def format_duration(seconds):
    hours, rest = divmod(int(round(seconds)), 3600)
    return f"{hours}:{rest // 60:02d}:{rest % 60:02d}"


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


def make_np_config(extra):
    # Targets and context are separate sets: the model joins them for the posterior.
    return NPConfig(x_dim=X_DIM, y_dim=Y_DIM, x_low=X_LOW, x_high=X_HIGH, target_includes_context=False,
                    use_latent=USE_LATENT, **{**NP_SETTINGS, **extra})


def plateau_scheduler(optimizer):
    # Stepped at each validation, so its patience is counted in validations.
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=max(1, PLATEAU_PATIENCE // VALIDATE_EVERY), min_lr=1e-6)


def best_validation(history, monitor, after_epoch):
    """The validation the model was restored to: the best `monitor` after `after_epoch`
    (or the last one, if there was none after it)."""
    epochs = history["val_epoch"]
    rows = [i for i, e in enumerate(epochs) if e > after_epoch] or [len(epochs) - 1]
    i = min(rows, key=lambda i: history[f"val_{monitor}"][i])
    best = {"best_epoch": epochs[i], "monitor": monitor}
    for name in ("elbo", "nll", "mse"):
        if f"val_{name}" in history:
            best[f"best_val_{name}"] = history[f"val_{name}"][i]
    return best


def save_results():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "results": results,
        "normalization": {"variables": VARIABLES, "y_mean": data.y_mean.tolist(), "y_std": data.y_std.tolist()},
        "settings": {"batch_size": BATCH_SIZE, "target_range": TARGET_RANGE, "steps_per_epoch": STEPS_PER_EPOCH,
                     "validation_batches": VALIDATION_BATCHES, "validate_every": VALIDATE_EVERY,
                     "epochs": EPOCHS, "plateau_patience": PLATEAU_PATIENCE,
                     "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                     "use_latent": USE_LATENT, "pooling": POOLING,
                     "np_learning_rate": NP_LEARNING_RATE, "kl_ramp_epochs": KL_RAMP_EPOCHS,
                     "np_monitor": NP_MONITOR, "test_lags": TEST_LAGS, "test_points": TEST_POINTS,
                     "test_batches": TEST_BATCHES, "test_mc_samples": TEST_MC_SAMPLES,
                     "seed": SEED, "device": str(device)},
    }
    with open(OUTPUT_DIR / "results.json", "w") as f:
        json.dump(payload, f, indent=2, default=str)


def train_one(name):
    """Builds and trains one model; returns it (with its best weights) and its history."""
    torch.manual_seed(SEED)
    if name in NP_MODELS:
        model_class, extra = NP_MODELS[name]
        model = model_class(make_np_config(extra)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=NP_LEARNING_RATE)
        history = train(model, train_batches, optimizer, EPOCHS, device, val_loader=val_batches,
                        validate_every=VALIDATE_EVERY, monitor=NP_MONITOR,
                        scheduler=plateau_scheduler(optimizer), warmup_epochs=KL_RAMP_EPOCHS,
                        kl_schedule=[e / KL_RAMP_EPOCHS for e in range(KL_RAMP_EPOCHS)] + [1.0],
                        grad_clip=GRAD_CLIP, early_stopping_patience=EARLY_STOPPING_PATIENCE,
                        checkpoint_dir=OUTPUT_DIR / name, print_every=PRINT_EVERY)
        info = {"config": asdict(model.cfg), "learning_rate": NP_LEARNING_RATE,
                **best_validation(history, NP_MONITOR, KL_RAMP_EPOCHS)}
    else:
        spec = BASELINES[name]
        model = spec["make"]().to(device)
        warmup = spec.get("mse_warmup", 0) if model.probabilistic else 0
        optimizer = torch.optim.Adam(model.parameters(), lr=spec["lr"])
        history = train_baseline(model, train_batches, optimizer, EPOCHS, device, val_batches,
                                 validate_every=VALIDATE_EVERY, scheduler=plateau_scheduler(optimizer),
                                 mse_warmup_epochs=warmup, grad_clip=GRAD_CLIP,
                                 early_stopping_patience=EARLY_STOPPING_PATIENCE,
                                 checkpoint_dir=OUTPUT_DIR / name, print_every=PRINT_EVERY)
        info = {"config": {"class": type(model).__name__, **model.kwargs}, "learning_rate": spec["lr"],
                **best_validation(history, "nll" if model.probabilistic else "mse", warmup)}
    return model, history, info


results, histories, models = {}, {}, {}

for name in RUN:
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    synchronize()
    start = time.perf_counter()
    model, history, info = train_one(name)
    synchronize()
    seconds = time.perf_counter() - start

    n_params, epochs_run = count_parameters(model), len(history["train_loss"])
    results[name] = {
        "parameters": n_params,
        "training_time_s": round(seconds, 1),
        "training_time": format_duration(seconds),
        "seconds_per_epoch": round(seconds / epochs_run, 2),
        "epochs_run": epochs_run,
        **info,
    }
    histories[name], models[name] = history, model
    save_results()  # after each model, so a crash in a later run keeps the earlier ones
    print(f"{name}: {n_params:,} parameters, trained in {format_duration(seconds)} ({epochs_run} epochs)")

## Results

In [ ]:
def fmt(value, width, spec=".4f"):
    return f"{'-':>{width}}" if value is None else f"{value:>{width}{spec}}"


header = (f"{'Model':<26}{'Parameters':>12}{'Training time':>15}{'s / epoch':>11}{'Epochs':>8}"
          f"{'Best epoch':>12}{'Val ELBO':>11}{'Val NLL':>11}{'Val MSE':>11}")
print(header)
print("-" * len(header))
for name, r in results.items():
    print(f"{name:<26}{r['parameters']:>12,}{r['training_time']:>15}{r['seconds_per_epoch']:>11.2f}"
          f"{r['epochs_run']:>8}{r['best_epoch']:>12}{fmt(r.get('best_val_elbo'), 11)}"
          f"{fmt(r.get('best_val_nll'), 11)}{fmt(r.get('best_val_mse'), 11)}")
print("Per value, standardized units, at the best epoch (chosen on: NPs " + NP_MONITOR
      + ", probabilistic baselines nll, deterministic ones mse).")
print("NP val NLL: from one latent sample of the prior; val ELBO: negative ELBO with KL weight 1"
      + ("." if USE_LATENT else " (without z both are the NLL)."))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for name, h in histories.items():
    epochs = range(1, len(h["train_loss"]) + 1)
    line, = axes[0].plot(epochs, h["train_loss"], label=name)
    color = line.get_color()
    if "val_nll" in h:
        axes[1].plot(h["val_epoch"], h["val_nll"], color=color, label=f"{name}: NLL")
    if "val_elbo" in h:
        axes[1].plot(h["val_epoch"], h["val_elbo"], color=color, ls="--", label=f"{name}: -ELBO")
    axes[2].plot(h["val_epoch"], h["val_mse"], color=color, label=name)
titles = ["Training loss (each model's own objective)", "Validation NLL / negative ELBO per value",
          "Validation MSE (standardized)"]
for ax, title in zip(axes, titles):
    ax.axvline(KL_RAMP_EPOCHS, color="gray", ls=":", lw=1)  # end of the NPs' KL ramp
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend(fontsize=7)
axes[2].set_yscale("log")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## Test set
With the best weights of each model, on the test trajectories:
1. the last time of every trajectory, all sensors, lags 10, 20 and 50, on a fixed set of `TEST_POINTS` grid
   points (drawn in proportion to their cell area, the same for every model);
2. the random-task test set: the same distribution as training and validation, drawn once with a fixed seed.

Per value (a value is one variable at one target), in standardized units, and the **RMSE** of the predictive
mean per variable in physical units:
- **NLL**: the mixture over the `TEST_MC_SAMPLES` latent samples taken for each value on its own, i.e. the
  marginal predictive density of one value (the product of these marginals ignores the correlations that z
  creates between targets). Probabilistic baselines: their Gaussian.
- **joint NLL**: the likelihood of the NP's predictive distribution of the whole target field,
  `log (1/S) sum_s prod_i p(y_i | z_s)`, divided by the number of values: one z explains all targets.
  A Monte Carlo estimate that is biased towards a worse (higher) NLL when S is small. For the
  probabilistic baselines (no latent) it equals the NLL.
- **90% coverage**: the fraction of true values inside mean +- 1.645 std of the predictive distribution
  (a Gaussian with its mean and variance; for the NPs, those of the mixture over latent samples). Close to
  0.90 is calibrated, lower is overconfident, higher underconfident. Probabilistic models only.
Every task is evaluated on its own, without the padding of its batch.

In [ ]:
def on_device(task):
    return {key: value.to(device) for key, value in task.items()}


def task_alone(batch, i):
    """Task i of a padded batch on its own: real context points, real targets, and the
    baselines' sensor history from the task's first reading on."""
    context, targets = ~batch["context_mask"][i], ~batch["target_mask"][i]
    first = int((batch["sensor_on"][i].sum(-1) > 0).nonzero()[0])
    task = {"x_context": batch["x_context"][i:i + 1, context], "y_context": batch["y_context"][i:i + 1, context],
            "x_target": batch["x_target"][i:i + 1, targets], "y_target": batch["y_target"][i:i + 1, targets],
            "target_index": batch["target_index"][i:i + 1, targets],
            "sensor_position": batch["sensor_position"][i:i + 1]}
    for key in ("sensor_history", "sensor_on", "sensor_time"):
        task[key] = batch[key][i:i + 1, first:]
    return task


@torch.no_grad()
def predict(model, task, n_samples=TEST_MC_SAMPLES):
    """One task (batch of 1, no padding), standardized units: predictive mean and variance per value
    (targets, variables), log-likelihood of the truth per value (mixture over the samples of each value on
    its own), and log-likelihood of all the values together (mixture over the samples of the product over
    the values). The last three are None for a deterministic model."""
    model.eval()
    task = on_device(task)
    y = task["y_target"][0]                                                  # (T, V)
    if isinstance(model, NeuralProcess):
        out = model(task["x_context"], task["y_context"], task["x_target"], num_samples=n_samples)
        log_p = out.y.log_prob(task["y_target"].unsqueeze(0))[:, 0]          # (S, T, V)
        means, variances = out.y.mean[:, 0], out.y.variance[:, 0]
        log_lik = torch.logsumexp(log_p, dim=0) - math.log(n_samples)
        joint = torch.logsumexp(log_p.sum((-2, -1)), dim=0) - math.log(n_samples)
        var = variances.mean(0) + means.var(0, unbiased=False)
        return means.mean(0), var, log_lik, joint
    mean, var = model(task)
    if var is None:
        return mean[0], None, None, None
    mean, var = mean[0], var[0]
    log_lik = -0.5 * (math.log(2 * math.pi) + var.log() + (y - mean) ** 2 / var)
    return mean, var, log_lik, log_lik.sum()


def metrics(model, tasks):
    """NLL, joint NLL and 90% coverage per value (None if deterministic) and RMSE per variable in physical
    units, over all targets of the tasks. Coverage: the fraction of truths inside mean +- 1.645 std of the
    predictive distribution (a Gaussian with the predictive mean and variance; for the NPs, of the mixture)."""
    nll_sum, joint_sum, covered, n_points, probabilistic = 0.0, 0.0, 0, 0, True
    se_sum = torch.zeros(Y_DIM, device=device)
    for task in tasks:
        task = on_device(task)
        mean, var, log_lik, joint = predict(model, task)
        y = task["y_target"][0]
        if log_lik is None:
            probabilistic = False
        else:
            nll_sum -= log_lik.sum().item()
            joint_sum -= joint.item()
            covered += ((y - mean).abs() <= 1.645 * var.sqrt()).sum().item()
        se_sum += (((mean - y) * Y_STD) ** 2).sum(0)
        n_points += y.shape[0]
    n_values = n_points * Y_DIM
    return {"nll": nll_sum / n_values if probabilistic else None,
            "joint_nll": joint_sum / n_values if probabilistic else None,
            "coverage_90": covered / n_values if probabilistic else None,
            "rmse": dict(zip(VARIABLES, (se_sum / n_points).sqrt().tolist()))}


test_targets = fixed_targets(data, TEST_POINTS, seed=SEED)
test_sets = {f"t_end_lag_{lag}": make_test_tasks(data, lag, test_targets) for lag in TEST_LAGS}
random_test_batches = make_random_batches(data, "test", TEST_BATCHES, batch_size=BATCH_SIZE, seed=SEED + 1,
                                          target_range=TARGET_RANGE)
test_sets["random_tasks"] = [task_alone(b, i) for b in random_test_batches for i in range(b["x_context"].shape[0])]
print({key: f"{len(tasks)} tasks" for key, tasks in test_sets.items()})

for name, model in models.items():
    torch.manual_seed(SEED)
    results[name]["test"] = {key: metrics(model, tasks) for key, tasks in test_sets.items()}

for key in test_sets:
    header = (f"{'Model':<26}{'NLL':>10}{'joint NLL':>12}{'cov 90%':>10}"
              + "".join(f"{'RMSE ' + v:>14}" for v in VARIABLES))
    print(f"\n{key}\n{header}")
    for name in models:
        m = results[name]["test"][key]
        print(f"{name:<26}{fmt(m['nll'], 10)}{fmt(m['joint_nll'], 12)}{fmt(m['coverage_90'], 10, '.3f')}"
              + "".join(f"{m['rmse'][v]:>14.4g}" for v in VARIABLES))
print(f"\n(std of y: {', '.join(f'{v} {s:.4g}' for v, s in zip(VARIABLES, data.y_std.tolist()))}; NLL in standardized units)")
save_results()
print(f"\nSaved to {OUTPUT_DIR / 'results.json'}")

### Example: one test trajectory
Truth, predictive mean, predictive standard deviation (probabilistic models) and absolute error of the variable
`PLOT_VARIABLE` at the last time of the first test trajectory, lag 20, on the fixed test points (longitude on
the x axis, colatitude on the y axis; the sensors are the red crosses).

In [ ]:
PLOT_VARIABLE = "h"
v = VARIABLES.index(PLOT_VARIABLE)
target_angles, sensor_angles = data.coordinates[test_targets].numpy(), data.coordinates[data.sensors].numpy()


def plot_field(ax, values, title, cmap="jet", vmin=None, vmax=None):
    mappable = ax.scatter(target_angles[:, 1], target_angles[:, 0], c=values.detach().cpu().numpy(), s=3,
                          cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(mappable, ax=ax, shrink=0.75)
    ax.scatter(sensor_angles[:, 1], sensor_angles[:, 0], color="red", s=14, marker="x")
    ax.invert_yaxis()
    ax.set_title(title)
    ax.axis("off")


example = test_sets["t_end_lag_20"][0]
truth = (example["y_target"][0, :, v] * data.y_std[v] + data.y_mean[v]).cpu()
fig, axes = plt.subplots(len(models), 4, figsize=(24, 4 * len(models)), squeeze=False)
torch.manual_seed(SEED)
for row, (name, model) in enumerate(models.items()):
    mean, var, _, _ = predict(model, example)
    mean = (mean[:, v] * Y_STD[v] + Y_MEAN[v]).cpu()
    vmin, vmax = min(truth.min(), mean.min()).item(), max(truth.max(), mean.max()).item()
    plot_field(axes[row, 0], truth, f"Truth ({PLOT_VARIABLE})", vmin=vmin, vmax=vmax)
    plot_field(axes[row, 1], mean, f"{name}: mean", vmin=vmin, vmax=vmax)
    if var is None:
        axes[row, 2].text(0.5, 0.5, "deterministic", ha="center", va="center", transform=axes[row, 2].transAxes)
        axes[row, 2].axis("off")
    else:
        plot_field(axes[row, 2], var[:, v].sqrt() * Y_STD[v], f"{name}: std", cmap="magma")
    plot_field(axes[row, 3], (mean - truth).abs(), f"{name}: |error|", cmap="magma")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"test_example_{PLOT_VARIABLE}_lag20.png", dpi=150)
plt.show()

### Latent samples of the Neural Processes
The predictive mean of single latent samples z ~ q(z | context), same task as above: how different the fields they explain are (only with `USE_LATENT = True`).

In [ ]:
N_SHOWN = 5
neural_processes = {name: m for name, m in models.items() if isinstance(m, NeuralProcess) and m.cfg.use_latent}
if neural_processes:
    fig, axes = plt.subplots(len(neural_processes), N_SHOWN, figsize=(5 * N_SHOWN, 4 * len(neural_processes)),
                             squeeze=False)
    torch.manual_seed(SEED)
    task = on_device(example)
    for row, (name, model) in enumerate(neural_processes.items()):
        model.eval()
        with torch.no_grad():
            out = model(task["x_context"], task["y_context"], task["x_target"], num_samples=N_SHOWN)
        samples = out.y.mean[:, 0, :, v] * Y_STD[v] + Y_MEAN[v]               # (N_SHOWN, targets)
        vmin, vmax = samples.min().item(), samples.max().item()
        for k in range(N_SHOWN):
            plot_field(axes[row, k], samples[k], f"{name}: sample {k + 1}", vmin=vmin, vmax=vmax)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"latent_samples_{PLOT_VARIABLE}_lag20.png", dpi=150)
    plt.show()